# T7 It3: does a second, better-aimed look give right places more lines?

Input: `results/second_look/<set>.csv` from `Second_Look_Run.ipynb` (GPU). For up to 5 candidates per photo (the std6 and wide27 choices and the
strongest other places, all >= 3 inliers) LoFTR looked again with the tile re-centred on the corrected place (`recentre`) and, in the smoke run,
with the query also re-drawn with the fitted heading and scale (`aim_p`, `aim_m`; the smoke run decides the sign).

Questions: (1) do right candidates gain lines and wrong ones not? (2) does the gain survive as a decision gain once the judge sees the second-look
signals (leave one set out, 0 wrong)? (3) what does it cost per photo?

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.rich_dump import load_rich
from train.checks.replay import error_sigma
RES = ROOT / "tests/Test_Iteration_7/results"
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
FOLDERS = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404", "nudt505": "nudt_fresh505"}
TAGS = [t for t in FOLDERS if (RES / "second_look" / (FOLDERS[t] + ".csv")).exists()]
print("sets with a second look:", TAGS)

def with_truth(name):
    sl = pd.read_csv(RES / "second_look" / (FOLDERS[name] + ".csv"))
    truth = {}
    for pid in sl.photo.unique():
        ph = load_rich(DUMPS / FOLDERS[name] / f"photo_{pid}.npz", with_matches=False)
        truth[pid] = ph
    sl["err0"] = [error_sigma(truth[p], x, y) for p, x, y in zip(sl.photo, sl.x0, sl.y0)]
    sl["err1"] = [error_sigma(truth[p], x, y) if n > 0 else e for p, x, y, n, e in zip(sl.photo, sl.sl_x, sl.sl_y, sl.sl_pl_inliers, sl.err0)]
    sl["data"] = name
    return sl
sl = pd.concat([with_truth(n) for n in TAGS], ignore_index=True)
sl["cls"] = np.where(sl.err0 <= 4, "right", np.where(sl.err0 > 8, "wrong", "mid"))
print(sl.groupby(["data", "variant", "cls"]).size().unstack())

sets with a second look: ['official', 'nudt101', 'nudt202', 'nudt404', 'nudt505']


cls                mid  right  wrong
data     variant                    
nudt101  aim_p     NaN  494.0  851.0
         recentre  NaN  494.0  851.0
nudt202  aim_p     2.0  324.0  598.0
         recentre  2.0  324.0  598.0
nudt404  aim_p     NaN  316.0  586.0
         recentre  NaN  316.0  586.0
nudt505  aim_p     1.0  498.0  867.0
         recentre  1.0  498.0  867.0
official aim_p     NaN  572.0  472.0
         recentre  NaN  572.0  472.0


## (1) Lines before and after, right vs wrong candidates

In [2]:
agg = sl[sl.cls != "mid"].groupby(["variant", "cls"]).agg(n=("photo", "size"), first=("first_inliers", "median"), second=("sl_pl_inliers", "median"),
     gain_mean=("sl_pl_inliers", lambda s: (s - sl.loc[s.index, "first_inliers"]).mean()),
     more=("sl_pl_inliers", lambda s: float((s > sl.loc[s.index, "first_inliers"]).mean())),
     seconds=("seconds", "mean"))
display(agg.round(3))
for v in sl.variant.unique():
    x = sl[(sl.variant == v) & (sl.cls != "mid")]
    b = pd.cut(x.first_inliers, [2, 5, 8, 12, 20, 1000])
    display(x.groupby([b, "cls"]).sl_pl_inliers.median().unstack().rename_axis(v))

n  first  second  gain_mean   more  seconds
variant  cls                                                  
aim_p    right  2204   48.0    45.0     -4.338  0.392    0.130
         wrong  3374    4.5     0.0     -3.230  0.089    0.136
recentre right  2204   48.0    49.0      0.591  0.422    0.140
         wrong  3374    4.5     0.0     -2.690  0.098    0.137

cls,right,wrong
recentre,,
"(2, 5]",4.0,0.0
"(5, 8]",6.0,4.0
"(8, 12]",9.0,8.0
"(12, 20]",16.0,12.0
"(20, 1000]",58.0,23.0


cls,right,wrong
aim_p,,
"(2, 5]",3.0,0.0
"(5, 8]",4.5,0.0
"(8, 12]",8.0,6.0
"(12, 20]",16.0,11.0
"(20, 1000]",54.0,23.0


## (2) Wrong candidates that reach the bar after the second look (the risk)

In [3]:
for v in sl.variant.unique():
    x = sl[(sl.variant == v)]
    print(v, "wrong candidates with >= 12 lines after:", int(((x.cls == "wrong") & (x.sl_pl_inliers >= 12)).sum()), "before:",
          int(((x.cls == "wrong") & (x.first_inliers >= 12)).sum()), "| right with >= 12 after:", int(((x.cls == "right") & (x.sl_pl_inliers >= 12)).sum()),
          "before:", int(((x.cls == "right") & (x.first_inliers >= 12)).sum()))

recentre wrong candidates with >= 12 lines after: 54 before: 65 | right with >= 12 after: 1958 before: 2012
aim_p wrong candidates with >= 12 lines after: 51 before: 65 | right with >= 12 after: 1846 before: 2012
